# Kinematics: Theta2 and Theta3

For any given Altitude and Roll pointing orientation, the inverse kinematics calculates the required Theta2 and Theta3 motor angles to achive the desired result. 

This notebook analyses the relationships between Altitude, Roll and Theta2, Theta3.


In [1]:
import pandas as pd

from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
import os, sys
import importlib
sys.path.insert(0, os.path.abspath(os.path.join('.', '..', 'driver')))
import kinematics 
importlib.reload(kinematics)
from kinematics import azaltroll_to_theta, theta_to_azaltroll, altitude_to_maxroll


Simulate a grid of pointing orientations spanning IK Altitude and Roll as well as FK Theta2 and Theta3 positions at a given Az

In [2]:

az = 180
d = []
for y in range(-80,81,5):
    for x in range(-80,81,5):
        f_t2 = x
        t1, t2, t3 = azaltroll_to_theta(az, x, y)
        _, alt, roll = theta_to_azaltroll(az, x, y)
        if x<-8:
            alt, roll, f_t2 = 0, 0, 0
        d.append({ 
            "i_alt": x, "i_roll": y, "i_theta2": t2, "i_theta3": t3,
            "f_alt": alt, "f_roll": roll, "f_theta2": f_t2, "f_theta3": y,
        })
d=pd.DataFrame(d)
d.columns


Index(['i_alt', 'i_roll', 'i_theta2', 'i_theta3', 'f_alt', 'f_roll',
       'f_theta2', 'f_theta3'],
      dtype='object')

# Altitude vs Theta2 and Theta3


In [3]:
fig = make_subplots(
    rows=1, 
    cols=2,
    subplot_titles=("Altitude vs Theta2", "Altitude vs Theta3")
)

# First plot (theta2)
fig1 = px.line(d, x="i_theta2", y="i_alt", color="i_roll")
for trace in fig1.data:
    trace.legendgroup = trace.name
    trace.showlegend = True   # Only show legend once
    fig.add_trace(trace, row=1, col=1)

# Second plot (theta3)
fig2 = px.line(d, x="i_theta3", y="i_alt", color="i_roll")
for trace in fig2.data:
    trace.legendgroup = trace.name
    trace.showlegend = False  # Hide duplicate legend entries
    fig.add_trace(trace, row=1, col=2)

# Adjust overall layout
fig.update_layout(height=800, width=1600, showlegend=True, legend_title_text="Roll (deg)")

fig.update_xaxes(title_text="Theta 3 (deg)", row=1, col=2)
fig.update_xaxes(title_text="Theta 2 (deg)", row=1, col=1)

fig.update_yaxes(title_text="Altitude (deg)", row=1, col=1)
fig.update_yaxes(title_text="Altitude (deg)", row=1, col=2)

fig.show()

# Roll vs Theta2 and Theta3

In [4]:
fig = make_subplots(
    rows=1, 
    cols=2,
    subplot_titles=("Roll vs Theta2", "Roll vs Theta3")
)

# First plot (theta2)
fig1 = px.line(d, x="i_theta2", y="i_roll", color="i_alt")
for trace in fig1.data:
    trace.legendgroup = trace.name
    trace.showlegend = True   # Only show legend once
    fig.add_trace(trace, row=1, col=1)

# Second plot (theta3)
fig2 = px.line(d, x="i_theta3", y="i_roll", color="i_alt")
for trace in fig2.data:
    trace.legendgroup = trace.name
    trace.showlegend = False  # Hide duplicate legend entries
    fig.add_trace(trace, row=1, col=2)

# Adjust overall layout
fig.update_layout(height=800, width=1600, showlegend=True, legend_title_text="Altitude (deg)")

fig.update_xaxes(title_text="Theta 3 (deg)", row=1, col=2)
fig.update_xaxes(title_text="Theta 2 (deg)", row=1, col=1)

fig.update_yaxes(title_text="Roll (deg)", row=1, col=1)
fig.update_yaxes(title_text="Roll (deg)", row=1, col=2)

fig.show()

# Altitude and Roll isobars in the Theta2/Theta3 space

In [5]:
fig = go.Figure()

# --- SOLID LINES: grouped by alt ---
for val in d["i_alt"].unique():
    df_sub = d[d["i_alt"] == val]
    fig.add_trace(
        go.Scatter(x=df_sub["i_theta3"], y=df_sub["i_theta2"], name=f"alt={val}",
            mode="lines", line=dict(dash="solid"), legendgroup="alt", showlegend=True)
    )

# --- DASHED LINES: grouped by roll ---
for val in d["i_roll"].unique():
    df_sub = d[d["i_roll"] == val]
    fig.add_trace(
        go.Scatter(x=df_sub["i_theta3"], y=df_sub["i_theta2"], name=f"roll={val}",
            mode="lines", line=dict(dash="dash"), legendgroup="roll", showlegend=True)
    )

fig.update_layout(
    height=900, width=1600, xaxis_title="Theta3 (deg)", yaxis_title="Theta2 (deg)", legend_title_text="Grouping"
)

fig.show()

# Theta2 and Theta3 isobars in the Altitude/Roll  space

In [6]:
fig = go.Figure()

# --- SOLID LINES: grouped by alt ---
for val in d["f_theta2"].unique():
    df_sub = d[d["f_theta2"] == val]
    fig.add_trace(
        go.Scatter(x=df_sub["f_roll"], y=df_sub["f_alt"], name=f"theta2={val}",
            mode="lines", line=dict(dash="solid"), legendgroup="alt", showlegend=True)
    )

# --- DASHED LINES: grouped by roll ---
for val in d["f_theta3"].unique():
    df_sub = d[d["f_theta3"] == val]
    fig.add_trace(
        go.Scatter(x=df_sub["f_roll"], y=df_sub["f_alt"], name=f"theta3={val}",
            mode="lines", line=dict(dash="dash"), legendgroup="roll", showlegend=True)
    )

fig.update_layout(
    height=900, width=1400, xaxis_title="Roll (deg)", yaxis_title="Altitude (deg)", legend_title_text="Grouping"
)

fig.show()

# Reachable Altitude and Roll

In [ ]:
import numpy as np
import plotly.graph_objects as go

# ── Functions copied verbatim from kinematics.py ─────────────────────────────
THETA2_MAX = 81.5  # hard mechanical limit for both alt and roll axes

def wrap_to_nearest(angle: float, target: float = 0.0) -> float:
    """Wrap angle to the value nearest to target, in steps of 360."""
    diff = angle - target
    diff = diff - 360 * round(diff / 360)
    return target + diff

def altitude_to_maxroll(alt_deg, theta2_max=THETA2_MAX):
    """Maximum achievable camera roll at a given sky altitude (kinematics.py)."""
    cos_ratio = np.cos(np.radians(theta2_max)) / np.cos(np.radians(alt_deg))
    if abs(cos_ratio) > 1:
        return 0.0
    return np.degrees(np.arccos(cos_ratio))

def reachable_azaltroll(az: float, alt: float, roll: float) -> tuple[float, float, float]:
    """Map any (az, alt, roll) to a mechanically reachable (az, alt, roll). (kinematics.py, verbatim)"""
    alt_norm = wrap_to_nearest(alt, target=0.0)
    roll_flip = 0.0
    if alt_norm > THETA2_MAX:
        alt_resolved = 180.0 - alt_norm
        az += 180.0
        roll_flip += 180.0
    elif alt_norm < -THETA2_MAX:
        alt_resolved = -180.0 - alt_norm
        az += 180.0
        roll_flip += 180.0
    else:
        alt_resolved = alt_norm
    alt_resolved = float(np.clip(alt_resolved, -THETA2_MAX, THETA2_MAX))
    az_resolved = az % 360.0
    max_roll = altitude_to_maxroll(alt_resolved, THETA2_MAX)
    roll_total = wrap_to_nearest(roll + roll_flip, target=0.0)
    if abs(roll_total) <= max_roll:
        roll_resolved = roll_total
    else:
        sign = 1.0 if roll_total >= 0 else -1.0
        roll_flipped = roll_total - sign * 180.0
        if abs(roll_flipped) <= max_roll:
            roll_resolved = roll_flipped
        else:
            roll_resolved = float(np.clip(roll_total, -max_roll, max_roll))
    return az_resolved, alt_resolved, roll_resolved

# ── Additional real hardware limits, also from kinematics.py ─────────────────
# q_to_theta() enforces an ASYMMETRIC valid theta2 range (theta2_min, theta2_max = -8, 83),
# not the symmetric +/-THETA2_MAX used by reachable_azaltroll's own alt clamp above.
# reachable_azaltroll alone will happily report e.g. alt=-50 as "reachable" (it just clips
# into +/-81.5), which is NOT physically true -- the real hardware stops at alt = -8.
ALT_HARD_MIN = -8.0
ALT_HARD_MAX = 83.0

# check_for_gimbal_lock() flags |theta2| < 1 (enter) / > 3 (exit) as a gimbal-lock zone.
# Near theta2 = 0 the alt and roll axes become numerically degenerate with azimuth, so
# roll authority collapses right around (alt, roll) = (0, 0) -- not across the whole
# alt = 0 line, just the small neighbourhood where theta2_required is tiny.
GIMBAL_LOCK_THETA2 = 3.0  # degrees, using the more conservative EXIT threshold

# ── Build the Alt x Roll grid and evaluate reachability via reachable_azaltroll ──
N_ALT, N_ROLL = 381, 361
alt_vals = np.linspace(-10, 85, N_ALT)
roll_vals = np.linspace(-85, 85, N_ROLL)
ALT, ROLL = np.meshgrid(alt_vals, roll_vals, indexing="ij")

REACHABLE = np.zeros_like(ALT, dtype=bool)
THETA2_REQ = np.full_like(ALT, np.nan)
for i in range(ALT.shape[0]):
    for j in range(ALT.shape[1]):
        a, r = ALT[i, j], ROLL[i, j]
        _, a_r, r_r = reachable_azaltroll(0.0, a, r)
        hw_ok = (ALT_HARD_MIN <= a <= ALT_HARD_MAX)
        REACHABLE[i, j] = hw_ok and np.isclose(a_r, a, atol=1e-6) and np.isclose(r_r, r, atol=1e-6)
        c = np.cos(np.radians(a)) * np.cos(np.radians(r))
        THETA2_REQ[i, j] = np.degrees(np.arccos(np.clip(c, -1, 1)))

GIMBAL_ZONE = THETA2_REQ < GIMBAL_LOCK_THETA2

# ── Colors (dark theme) ───────────────────────────────────────────────────────
BG          = "#0e1117"
PBG         = "#331f1f"
GRID_CLR    = "rgba(255,255,255,0.08)"
TEXT_CLR    = "#e6e6e6"
GREEN_DIM   = "rgba(51,209,122,0.35)"
GREEN_LABEL = "#33d17a"
FLOOR_LINE  = "#ff8a3d"
LOCK_CLR    = "#ff8a3d"
ISO_LINE    = "rgba(255,255,255,0.35)"

fig = go.Figure()

# 1) Reachable region fill (as actually returned by reachable_azaltroll + hardware floor/ceiling)
Z_REACH = np.where(REACHABLE, 1.0, np.nan)
fig.add_trace(go.Heatmap(
    x=roll_vals, y=alt_vals, z=Z_REACH,
    zmin=0, zmax=1,
    colorscale=[[0, GREEN_DIM], [1, GREEN_DIM]],
    showscale=False, hoverinfo="skip", zsmooth=False,
))

# 2) theta2-required iso-contours beyond the limit, labeled with their degree value
fig.add_trace(go.Contour(
    x=roll_vals, y=alt_vals, z=THETA2_REQ,
    contours=dict(start=THETA2_MAX, end=130, size=10, coloring="none",
                  showlabels=True, labelfont=dict(size=11, color=ISO_LINE)),
    line=dict(color=ISO_LINE, width=1, dash="dot"),
    showscale=False, hoverinfo="skip",
))

# 3) Hard theta2 = THETA2_MAX boundary curve
fig.add_trace(go.Contour(
    x=roll_vals, y=alt_vals, z=THETA2_REQ,
    contours=dict(start=THETA2_MAX, end=THETA2_MAX, size=1, coloring="none",
                  showlabels=True, labelfont=dict(size=22, color=FLOOR_LINE), labelformat=".1f"),
    line=dict(color=FLOOR_LINE, width=3),
    showscale=False, hoverinfo="skip",
))

# 4) Gimbal-lock hazard zone near (alt, roll) = (0, 0)
fig.add_trace(go.Contour(
    x=roll_vals, y=alt_vals, z=np.where(GIMBAL_ZONE, 1.0, 0.0),
    contours=dict(start=0.5, end=0.5, size=1, coloring="none"),
    line=dict(color=LOCK_CLR, width=2, dash="dash"),
    showscale=False, hoverinfo="skip",
))

# 5) Hard altitude floor at -8 deg (real hardware stop, independent of the theta2 curve)
fig.add_hline(y=ALT_HARD_MIN, line=dict(color=FLOOR_LINE, width=2, dash="dot"))
fig.add_hline(y=0, line=dict(color="rgba(255,255,255,0.25)", width=1))
fig.add_vline(x=0, line=dict(color="rgba(255,255,255,0.25)", width=1))

fig.update_layout(
    title=dict(
        text="Benro Polaris — Reachable Alt / Roll Envelope",
        x=0.02, xanchor="left",
        font=dict(color=TEXT_CLR, size=42, family="Arial, sans-serif"),
    ),
    paper_bgcolor=BG, plot_bgcolor=PBG,
    font=dict(color=TEXT_CLR, family="Arial, sans-serif", size=22),
    width=1280, height=800,
    margin=dict(l=90, r=60, t=110, b=80),
    xaxis=dict(title="Roll (deg)", range=[-90, 90], dtick=20,
               gridcolor=GRID_CLR, zeroline=False, showline=True, linecolor="rgba(255,255,255,0.3)",
               ticks="outside", tickcolor="rgba(255,255,255,0.3)"),
    yaxis=dict(title="Altitude (deg)", range=[-10, 90], dtick=10,
               gridcolor=GRID_CLR, zeroline=False, showline=True, linecolor="rgba(255,255,255,0.3)",
               ticks="outside", tickcolor="rgba(255,255,255,0.3)"),
    annotations=[
        dict(x=0, y=45, xref="x", yref="y", text="reachable envelope", showarrow=False,
             font=dict(color=GREEN_LABEL, size=42)),
        dict(x=81.5, y=15, xref="x", yref="y",
             text=f"\u03b8\u2082 = {THETA2_MAX:.1f}\u00b0 boundary", showarrow=True, ax=-120, ay=-60,
             font=dict(color=FLOOR_LINE, size=22), arrowcolor=FLOOR_LINE),
        dict(x=-40, y=ALT_HARD_MIN, xref="x", yref="y",
             text=f"hard floor: alt = {ALT_HARD_MIN:.0f}\u00b0", showarrow=True, ax=-30, ay=-120,
             font=dict(color=FLOOR_LINE, size=22), arrowcolor=FLOOR_LINE),
        dict(x=25, y=6, xref="x", yref="y",
             text="gimbal lock zone", showarrow=True,
             ax=-40, ay=-10, font=dict(color=LOCK_CLR, size=22), arrowcolor=LOCK_CLR),
    ],
)

fig.show()

In [7]:
max_theta2 = 81.5
alts = list(range(-8, 90, 2))
max_rolls = [altitude_to_maxroll(a, max_theta2) for a in alts]


fig = go.Figure()

fig.add_trace(go.Scatter(
    x=max_rolls2,
    y=alts,
    mode='lines+markers',
    name=f'Max Roll Angle°)',
    line=dict(color='green', width=2),
    marker=dict(size=5),
    hovertemplate='Alttitude: %{y}°<br>Max Roll Angle: %{x:.1f}°<extra></extra>'
))


fig.update_layout(
    title=f'Maximum achievable Roll vs Altitude (limited by Theta2 = {max_theta2}°)',
    xaxis_title='Max Roll Angle (deg)',
    yaxis_title='Altitude (deg)',
    xaxis=dict(range=[0, 85], dtick=10),
    yaxis=dict(range=[-10, 92], dtick=10, zeroline=True),
    width=800,
    height=500,
    template='plotly_dark',
    hovermode='x unified'
)

NameError: name 'max_rolls2' is not defined